In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

In [ ]:
df = pd.read_csv("/content/creditcard.csv")

/tmp/ipykernel_4185/3469698040.py:1: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/creditcard.csv")


In [ ]:
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0.0
1,0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0.0
2,1,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0.0
3,1,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0.0
4,2,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0.0


In [ ]:
df.isnull().sum()

,0
Time,0
V1,0
V2,0
V3,0
V4,0
V5,0
V6,0
V7,0
V8,0
V9,0


In [ ]:
X = df.drop("Class", axis = 1).values
y = df["Class"].values

In [ ]:
X_train , X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)

In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(27008, 30)
(6752, 30)
(27008,)
(6752,)


In [ ]:
X_train_normal = X_train[y_train == 0]

In [ ]:
print(X_train_normal)

[[34434 1.27894684550214 0.284826797688769 ... -0.004258822725526
  0.0099855109392026 0.89]
 [22921 0.728480605725101 -0.971568908631562 ... -0.0664974769877725
  0.0220632784495304 189.09]
 [29985 1.15038246360397 0.0805700509290016 ... -0.0820710016905988
  0.0054634478869851 52.9]
 ...
 [19619 -0.247756404735851 1.25430937607771 ... 0.137655220109064
  0.143734915788889 14.49]
 [654 -0.833568321319693 0.606174187718198 ... 0.164383984556828
  0.274361005101437 9.9]
 [27235 0.995760270134647 0.329195722084344 ... -0.0659275184903826
  0.0153932098787226 113.75]]


In [ ]:
scaler = StandardScaler()
X_train_normal = scaler.fit_transform(X_train_normal)
X_test = scaler.transform(X_test)

In [ ]:
input_dim = X_train_normal.shape[1]

In [ ]:
model = Sequential([
    Input(shape=(input_dim, )),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(16, activation= 'relu'),
    Dense(input_dim, activation = "linear")
])

In [ ]:
model.compile(
    optimizer = "adam",
    loss = "mse"
)

In [ ]:
model.fit(X_train_normal, X_train_normal, epochs=1, batch_size=32, shuffle=True)

842/842 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 0.8042


In [ ]:
predictions = model.predict(X_test)

211/211 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [ ]:
errors = np.mean(np.square(X_test - predictions), axis=1)

# 11. Set anomaly threshold
threshold = np.percentile(errors, 95)

# 12. Detect anomalies
y_pred = (errors > threshold).astype(int)

# 13. Evaluate model
print("Threshold:", threshold)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Classification Report:")
print(classification_report(y_test, y_pred))

Threshold: 1.2867761578363592
Confusion Matrix:
[[6414  319]
 [   0   19]]
Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      0.95      0.98      6733
         1.0       0.06      1.00      0.11        19

    accuracy                           0.95      6752
   macro avg       0.53      0.98      0.54      6752
weighted avg       1.00      0.95      0.97      6752

